In [1]:
using SpecialFunctions
using LinearAlgebra
using Plots, LaTeXStrings
using Struve
using KrylovKit
using SparseArrays
using Arpack  # For sparse eigenvalue computations
using QuadGK
using ForwardDiff
using FFTW
using Statistics
using Measures
using Base.Threads
using TensorOperations
using CSV, DataFrames
include("module_mos2_plot3.jl")
using .exciton

In [2]:
Nsample=60
sz = 1
lattice = TMDLattice();
VInt = InteractionMatrix(lattice, Nsample; lambda = 0.667, r0 = 4.288)
bi_1 = lattice.b1 / Nsample
bi_2 = lattice.b2 / Nsample
bi_3 = -bi_1 - bi_2
wb = 2 / (3 * norm(bi_1)^2)

strainlist = -6:0.5:6

wannierlist = Vector{ComplexF64}(undef, length(strainlist))
blochlist = Vector{Vector{ComplexF64}}(undef, length(strainlist))

# Parallelize the outer loop
Threads.@threads for i in 1:length(strainlist)
	strain_val = strainlist[i]
	# --- Everything below is local to this specific thread/strain ---
	epsilonyy = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5
	println("Thread $(Threads.threadid()) doing strain $strain_val (Index $i)")
	flush(stdout)
	# 1. Initialize System
	# Note: 'lattice', 'Nsample', 'sz' must be defined in global scope
	local_sys = TMDBSE(lattice, [0.0, 0], Nsample; sz, epsilonyy)
	local_sys = add_bse_kernel(local_sys, VInt)

	bsemat = local_sys.BSEKernel
	xlen = size(bsemat)[1]
	x0 = rand(xlen) # Random starting vector

	valslist, vecslist, info = eigsolve(bsemat, x0, 10, :SR,
		krylovdim = 40, tol = 1e-10, maxiter = 20,
		verbosity = 0, ishermitian = true)

	local_sys = add_wannier(local_sys)
	psik = vecslist[1]
	psi_real = exciton.envelope_real_fft(psik, local_sys)
	r1 = Rshift_calculator(local_sys, psi_real, div(Nsample, 2) - 2)

	wannierlist[i] = r1[2]
	flush(stdout)

	wv, wc1, wc2 = local_sys.Wannier.valence, local_sys.Wannier.conduction1, local_sys.Wannier.conduction2

	blochlist[i] = exciton.exciton_subroutine_4th(wv, wc1, wc2; state = 1, polarization = [[cos(2pi / 3), sin(2pi / 3)], [cos(3pi / 4), sin(3pi / 4)], [cos(5pi / 6), sin(5pi / 6)]], VInt, lattice, epsilonyy, sz = 1, Nsample)
end

polarization_list=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list[i]=real(blochlist[i][1])
end

polarization_list2=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list2[i]=real(blochlist[i][2])
end

polarization_list3=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list3[i]=real(blochlist[i][3])
end

sigma1=polarization_list .- real.(wannierlist)
println("std1 is ", sqrt(sum(sigma1 .^ 2 ./ 25)))
sigma2=polarization_list2 .- real.(wannierlist)
println("std2 is ", sqrt(sum(sigma2 .^ 2 ./ 25)))
sigma3=polarization_list3 .- real.(wannierlist)
println("std3 is ", sqrt(sum(sigma3 .^ 2 ./ 25)))

Thread 3 doing strain -2.5 (Index 8)
Thread 1 doing strain -6.0 (Index 1)
Thread 4 doing strain 0.5 (Index 14)
Thread 2 doing strain 3.5 (Index 20)
gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
[3.0151634177868515e-7, 0.0019326979106490312]
[1.631120875875705e-7, 0.0002852699908554002]
[1.1362470161904361e-7, -0.0013644937695109626]
[1.0841132296611454e-7, -0.0028666240152075504]
Center Energy: 1.0211527929476145
Center Energy: 1.2310283624621807
Center Energy: 1.4287942295912506
Center Energy: 1.644843253455837
rtot for all polarizations: ComplexF64[0.0019323726124410598 + 2.1131997866239147e-8im, 0.001932376679097977 + 2.1131997866239147e-8im, 0.0019323806173979366 + 2.1131997866239147e-8im]
Thread 3 doing strain 4.0 (Index 21)
gauge fixing successful!
rtot for all polarizations: ComplexF64[0.0002848504406487965 + 2.0511130609852432e-8im, 0.0002848565815835308 + 2.0511130609852432e-8im, 0.0002848627513480825 + 2.0511130609852432e-

In [3]:
df = DataFrame(
    strainlist = strainlist,
    rshift_wannier = real.(wannierlist),
    rshift_full_2pi_3 = polarization_list,
    rshift_full_3pi_4 = polarization_list2,
    rshift_full_5pi_6 = polarization_list3
)

CSV.write("fig_s2.csv", df)

"fig_s2.csv"